In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.tri as tri
from sklearn.externals.array_api_compat.dask.array import asarray

# Définir les sommets du triangle
A = np.array([0, 0])
B = np.array([2, 1])
C = np.array([1, 2])

triangle = np.array([A, B, C])

def barycentric_coords(P, A, B, C):
    denom = (B[1] - C[1]) * (A[0] - C[0]) + (C[0] - B[0]) * (A[1] - C[1])
    lambda_A = ((B[1] - C[1]) * (P[0] - C[0]) + (C[0] - B[0]) * (P[1] - C[1])) / denom
    lambda_B = ((C[1] - A[1]) * (P[0] - C[0]) + (A[0] - C[0]) * (P[1] - C[1])) / denom
    lambda_C = 1 - lambda_A - lambda_B # not symmmetric...

    return lambda_A, lambda_B, lambda_C

def barycentric_coords_3d_triangle(P, A, B, C):
    """
    Compute barycentric coordinates of a point P with respect to a triangle ABC in 3D.

    Parameters:
    - P: Point (x, y, z) in 3D.
    - A, B, C: Vertices of the triangle in 3D.

    Returns:
    - (lambda_A, lambda_B, lambda_C): Barycentric coordinates.
    """
    A = np.array(A)
    B = np.array(B)
    C = np.array(C)
    P = np.array(P)

    # Vectors
    AC = C - A
    AB = B - A
    AP = P - A

    # Compute dot products for the normal vector and projections
    AC2 = np.dot(AC, AC)
    dot01 = np.dot(AC, AB)
    dot02 = np.dot(AC, AP)
    AB2 = np.dot(AB, AB)
    dot12 = np.dot(AB, AP)

    denom = AC2 * AB2 - dot01 * dot01
    if abs(denom) < 1e-10:
        # Triangle is degenerate (area ~ 0)
        return None  # or raise an error

    lambda_C = (AC2 * dot12 - dot01 * dot02) / denom
    lambda_B = (dot01 * dot12 - dot02 * AB2) / denom
    lambda_A = 1 - lambda_B - lambda_C

    return lambda_A, lambda_B, lambda_C


# Créer une grille de points dans le triangle
def generate_points_in_triangle(A, B, C, n=50):
    # Génère des points aléatoires dans le rectangle englobant
    min_x, max_x = min(A[0], B[0], C[0]), max(A[0], B[0], C[0])
    min_y, max_y = min(A[1], B[1], C[1]), max(A[1], B[1], C[1])
    points = np.random.rand(n, 2) * np.array([max_x - min_x, max_y - min_y]) + np.array([min_x, min_y])

    # Filtrer les points à l'intérieur du triangle
    inside = []
    for P in points:
        lambda_A, lambda_B, lambda_C = barycentric_coords(P, A, B, C)
        if lambda_A >= 0 and lambda_B >= 0 and lambda_C >= 0:
            inside.append(P)
    return np.array(inside)

# Générer les points
points = generate_points_in_triangle(A, B, C, n=2000)

# Calculer les coordonnées barycentriques pour chaque point
lambda_A = np.zeros(len(points))
lambda_B = np.zeros(len(points))
lambda_C = np.zeros(len(points))

for i, P in enumerate(points):
    lambda_A[i], lambda_B[i], lambda_C[i] = barycentric_coords(P, A, B, C)

def plot_triangle(points, lambdas, triangle):

    # Tracer les 3 figures
    fig, (ax1, ax2, ax3) = plt.subplots(1, 3, figsize=(15, 5))

    # Tracer le triangle
    def plot_triangle(ax, A, B, C):
        ax.plot([A[0], B[0]], [A[1], B[1]], 'k-')
        ax.plot([B[0], C[0]], [B[1], C[1]], 'k-')
        ax.plot([C[0], A[0]], [C[1], A[1]], 'k-')
        ax.scatter([A[0], B[0], C[0]], [A[1], B[1], C[1]], c='red', s=50)
        ax.text(A[0], A[1], 'A', fontsize=12, ha='right')
        ax.text(B[0], B[1], 'B', fontsize=12, ha='right')
        ax.text(C[0], C[1], 'C', fontsize=12, ha='right')

    # Tracer lambda_A
    sc1 = ax1.scatter(points[:, 0], points[:, 1], c=lambdas[0], cmap='Reds', s=5)
    plot_triangle(ax1, triangle[0], triangle[1], triangle[2])
    ax1.set_title(r'$\lambda_A$ (Poids de A)')
    fig.colorbar(sc1, ax=ax1, label='Valeur')

    # Tracer lambda_B
    sc2 = ax2.scatter(points[:, 0], points[:, 1], c=lambdas[1], cmap='Blues', s=5)
    plot_triangle(ax2, triangle[0], triangle[1], triangle[2])
    ax2.set_title(r'$\lambda_B$ (Poids de B)')
    fig.colorbar(sc2, ax=ax2, label='Valeur')

    # Tracer lambda_C
    sc3 = ax3.scatter(points[:, 0], points[:, 1], c=lambdas[2], cmap='Greens', s=5)
    plot_triangle(ax3, triangle[0], triangle[1], triangle[2])
    ax3.set_title(r'$\lambda_C$ (Poids de C)')
    fig.colorbar(sc3, ax=ax3, label='Valeur')

    plt.tight_layout()
    plt.show()

plot_triangle([lambda_A,lambda_B,lambda_C], triangle)


In [ ]:
nb = 11
nb_pts = nb * (nb+1) / 2
uvw_points = np.asarray([(u, v, 1-u-v) for u in np.linspace(0,1,nb) for v in np.linspace(0,1,nb) if u + v <= 1])
print(uvw_points)

In [ ]:
print(np.shape(uvw_points))

In [ ]:
lambdas = uvw_points.T
points = np.asarray([(lambdas[0][i]*triangle[0] + lambdas[1][i]*triangle[1] + lambdas[2][i]*triangle[2]) for i in range(lambdas.shape[])])
print(points)


In [ ]:
plot_triangle(lambdas.T, triangle)